In [ ]:
from dataclasses import dataclass, field
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
from aicsimageio import AICSImage
from desmin_reconstruction.plotting import save_panel
from desmin_reconstruction.preprocessing.bbox import BoundingSquare
from IPython.display import display, SVG
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.ticker import FormatStrFormatter


COLOR_DICT = {
    "desmin": "#CC3311",
    "actin": "#0077BB",
    "actinin": "#009988",
    "myosin": "#BBBBBB",
}

datadir = Path("../../data/SIM/actin")

In [ ]:
@dataclass
class PreparedSIM:
    protein: str
    grids: dict
    scalar_names: dict = field(default_factory=dict)
    crop_slices: tuple | None = None
    intensity_divisor: float | None = None


@dataclass
class RenderedSIM:
    protein: str
    image: np.ndarray
    extent: tuple
    image_extent: tuple
    world_bounds: tuple
    colors: dict


def pair_from_grids(desmin_grid, actin_grid, *, scalar_names=("data", "data")):
    """Reuse existing registered grids and their current intensity values."""
    if len(scalar_names) != 2:
        raise ValueError("Supply one scalar-array name per channel.")
    for grid, scalar in zip((desmin_grid, actin_grid), scalar_names):
        if scalar not in grid.array_names:
            raise ValueError(f"Grid does not contain scalar array {scalar!r}.")
    if not np.allclose(desmin_grid.bounds, actin_grid.bounds):
        raise ValueError("The registered grids must have matching physical bounds.")
    return PreparedSIM(
        "actin",
        {"desmin": desmin_grid, "actin": actin_grid},
        dict(zip(("desmin", "actin"), scalar_names)),
    )


def prepare_xy_pair(
    img_data,
    pixel_sizes,
    *,
    bbox=None,
    desmin_channel=0,
    actin_channel=1,
    intensity_divisor=None,
):
    """Crop and normalize exactly as in the supplied grid-construction code.

    img_data is C,X,Y,Z; pixel_sizes is (dx,dy,dz) in micrometers. No transpose,
    flip, maximum projection, or per-channel contrast normalization is applied.
    bbox is an object with x0/x1/y0/y1, or a tuple (x0,x1,y0,y1) in micrometers.
    Upper slice indices include the cell containing x1/y1, as in the original.
    Grid origins remain (0,0,0), so axes show local ROI coordinates.

    Integer inputs are divided by np.iinfo(img_data.dtype).max. For floating
    inputs, pass an explicit divisor (1 for already normalized data).
    """
    import pyvista as pv

    data = np.asarray(img_data)
    spacing = np.asarray(pixel_sizes, dtype=float)
    if data.ndim != 4 or any(n == 0 for n in data.shape):
        raise ValueError("img_data must have nonempty C,X,Y,Z dimensions.")
    if spacing.shape != (3,) or not np.isfinite(spacing).all() or np.any(spacing <= 0):
        raise ValueError(
            "pixel_sizes must be positive finite (dx,dy,dz) in micrometers."
        )
    if desmin_channel == actin_channel:
        raise ValueError("Choose distinct Desmin and Actin channels.")
    for channel in (desmin_channel, actin_channel):
        if not 0 <= channel < data.shape[0]:
            raise ValueError("Channel index is outside img_data.")
    if intensity_divisor is None:
        if not np.issubdtype(data.dtype, np.integer):
            raise ValueError("For floating data, specify intensity_divisor explicitly.")
        intensity_divisor = np.iinfo(data.dtype).max
    intensity_divisor = float(intensity_divisor)
    if not np.isfinite(intensity_divisor) or intensity_divisor <= 0:
        raise ValueError("intensity_divisor must be finite and positive.")
    nx, ny, nz = data.shape[1:]
    if bbox is None:
        xs, ys = slice(0, nx), slice(0, ny)
    else:
        xy = [bbox.x0, bbox.x1, bbox.y0, bbox.y1] if hasattr(bbox, "x0") else bbox
        xy = np.asarray(xy, dtype=float)
        if xy.shape != (4,) or not np.isfinite(xy).all():
            raise ValueError("bbox must supply finite (x0,x1,y0,y1).")
        x0, x1, y0, y1 = xy
        if not (0 <= x0 < x1 <= nx * spacing[0] and 0 <= y0 < y1 <= ny * spacing[1]):
            raise ValueError("bbox is outside the physical image bounds.")
        xs = slice(int(x0 // spacing[0]), min(nx, int(x1 // spacing[0] + 1)))
        ys = slice(int(y0 // spacing[1]), min(ny, int(y1 // spacing[1] + 1)))
    grids = {}
    for name, channel in (("desmin", desmin_channel), ("actin", actin_channel)):
        cropped = data[channel, xs, ys, :]
        if not np.isfinite(cropped).all():
            raise ValueError(f"{name}: nonfinite image values.")
        grid = pv.ImageData(
            dimensions=np.array(cropped.shape) + 1, spacing=spacing, origin=(0, 0, 0)
        )
        grid.cell_data[name] = cropped.ravel(order="F") / intensity_divisor
        grids[name] = grid
    return PreparedSIM(
        "actin",
        grids,
        {"desmin": "desmin", "actin": "actin"},
        (xs, ys, slice(0, nz)),
        intensity_divisor,
    )


def render_sim_pair(
    prepared,
    *,
    channels=None,
    colors=None,
    height_px=1400,
    clim=(0.0, 0.2),
    opacity=(0.0, 0.1, 0.8),
    color_modes=None,
):
    """Generalization of the previous renderer; select one or both channels.

    Defaults to CONSTANT color for both Desmin and Actin. All views use the
    same calibrated XY camera, opacity, intensity limits, linear interpolation,
    and maximum volume blending. Overlays contain two PyVista volume actors.
    Also accepts the previous PreparedSIM object with protein/grids fields.
    """
    import pyvista as pv

    channels = ("desmin", prepared.protein) if channels is None else tuple(channels)
    if not channels or len(channels) != len(set(channels)):
        raise ValueError("channels must contain one or more distinct names.")
    colors = {**COLOR_DICT, **(colors or {})}
    modes = {name: "constant" for name in prepared.grids}
    modes.update(color_modes or {})
    scalar_names = getattr(prepared, "scalar_names", {}) or {}
    height_px = int(height_px)
    if height_px < 2:
        raise ValueError("height_px must be at least 2.")
    limits = np.asarray(clim, dtype=float)
    opacity = np.asarray(opacity, dtype=float)
    if limits.shape != (2,) or not np.isfinite(limits).all() or limits[0] >= limits[1]:
        raise ValueError("clim must contain two increasing finite values.")
    if (
        opacity.ndim != 1
        or opacity.size < 2
        or not np.isfinite(opacity).all()
        or np.any((opacity < 0) | (opacity > 1))
    ):
        raise ValueError("opacity must contain at least two finite values in [0,1].")
    bounds = np.asarray(prepared.grids["desmin"].bounds, dtype=float)
    for name in channels:
        if name not in prepared.grids or name not in colors:
            raise ValueError(f"Missing grid or color for {name!r}.")
        if not np.allclose(prepared.grids[name].bounds, bounds):
            raise ValueError("All channels must have the same physical bounds.")
        if modes[name] not in ("constant", "ramp"):
            raise ValueError("Color modes must be 'constant' or 'ramp'.")
    x0, x1, y0, y1, z0, z1 = bounds
    dx, dy = x1 - x0, y1 - y0
    if min(dx, dy) <= 0:
        raise ValueError("Grids must have positive X/Y dimensions.")
    width_px = max(2, int(np.ceil(height_px * dx / dy)))
    cx, cy, cz = (x0 + x1) / 2, (y0 + y1) / 2, (z0 + z1) / 2
    distance = 2 * max(dx, dy, z1 - z0)
    half_width = dy / 2 * width_px / height_px
    p = pv.Plotter(off_screen=True, window_size=(width_px, height_px))
    try:
        for name in channels:
            cmap = (
                [colors[name]]
                if modes[name] == "constant"
                else LinearSegmentedColormap.from_list(
                    name + "_ramp", ["black", colors[name]]
                )
            )
            actor = p.add_volume(
                prepared.grids[name],
                scalars=scalar_names.get(name, name),
                cmap=cmap,
                opacity=opacity.tolist(),
                clim=tuple(limits),
                blending="maximum",
                show_scalar_bar=False,
                reset_camera=False,
            )
            actor.prop.interpolation_type = "linear"
        p.set_background("black")
        p.enable_parallel_projection()
        p.camera_position = [(cx, cy, cz + distance), (cx, cy, cz), (0, 1, 0)]
        p.camera.parallel_scale = dy / 2
        p.reset_camera_clipping_range()
        rgb = np.array(p.screenshot(transparent_background=False), copy=True)
    finally:
        p.close()
    return RenderedSIM(
        prepared.protein,
        rgb,
        (0.0, dx, 0.0, dy),
        (cx - half_width - x0, cx + half_width - x0, 0.0, dy),
        tuple(bounds),
        colors,
    )


def render_sim_triptych(prepared, **render_kwargs):
    """Return cached Desmin, Actin, and two-actor overlay screenshots."""
    if prepared.protein != "actin":
        raise ValueError("This triptych expects Desmin and Actin.")
    return {
        key: render_sim_pair(prepared, channels=names, **render_kwargs)
        for key, names in (
            ("desmin", ("desmin",)),
            ("actin", ("actin",)),
            ("overlay", ("desmin", "actin")),
        )
    }


def add_zline_guide(
    ax,
    *,
    start=(0.74, 0.035),
    end=(0.98, 0.275),
    n_ticks=7,
    tick_length=0.005,
    text_position=(0.995, 0.045),
    text="Z lines",
    fontsize=11,
    color="white",
    linewidth=0.8,
    coords="axes",
):
    """Editable schematic guide; coordinates are axes fractions or data (um).

    Positions are approximate matches to the reference, not detected Z lines.
    For feature alignment use coords='data' and your measured positions.
    tick_offset is in the same units as start/end.
    """
    transform = ax.transAxes if coords == "axes" else ax.transData
    if coords not in ("axes", "data") or int(n_ticks) < 2:
        raise ValueError("Use coords='axes'/'data' and at least two ticks.")
    points = np.linspace(start, end, int(n_ticks))
    artists = ax.plot(
        points[:, 0],
        points[:, 1],
        color=color,
        lw=linewidth,
        transform=transform,
        clip_on=False,
        zorder=5,
    )
    for point in points:
        tick_angle = np.atan2((end[0] - start[0]), -(end[1] - start[1]))
        tip = point + np.asarray(
            (tick_length * np.cos(tick_angle), tick_length * np.sin(tick_angle))
        )
        artists += ax.plot(
            [point[0], tip[0]],
            [point[1], tip[1]],
            color=color,
            lw=linewidth,
            transform=transform,
            clip_on=False,
            zorder=5,
        )
    artists.append(
        ax.text(
            *text_position,
            text,
            transform=transform,
            color=color,
            fontsize=fontsize,
            ha="right",
            va="bottom",
            clip_on=False,
            zorder=6,
        )
    )
    return artists


def add_distribution_guide(
    ax,
    *,
    text="Horizontal distribution of desmin IF",
    text_position=(0.62, 1.035),
    arrow_tips=((0.66, 1.015), (0.73, 1.015), (0.985, 1.015)),
    tail_offset=(5, 7),
    fontsize=10,
    color="#CC3311",
    mutation_scale=10,
    linewidth=1.1,
    coords="axes",
):
    """Editable text/arrows above the overlay. Tail offsets are in points.

    text_position/arrow_tips use axes fractions by default; coords='data' uses
    local micrometers. Larger tail_offset lengthens tails; mutation_scale
    enlarges arrowheads. Defaults approximate the supplied reference only.
    """
    if coords not in ("axes", "data"):
        raise ValueError("Use coords='axes' or 'data'.")
    transform = ax.transAxes if coords == "axes" else ax.transData
    artists = [
        ax.text(
            *text_position,
            text,
            transform=transform,
            ha="right",
            va="bottom",
            color=color,
            fontsize=fontsize,
            clip_on=False,
            zorder=6,
        )
    ]
    for tip in arrow_tips:
        artists.append(
            ax.annotate(
                "",
                xy=tip,
                xycoords=transform,
                xytext=tail_offset,
                textcoords="offset points",
                annotation_clip=False,
                zorder=6,
                arrowprops=dict(
                    arrowstyle="->",
                    color=color,
                    lw=linewidth,
                    mutation_scale=mutation_scale,
                    shrinkA=0,
                    shrinkB=0,
                    clip_on=False,
                ),
            )
        )
    return artists


def compose_sim_triptych(
    views,
    *,
    panel_height=2.6,
    panel_gap=0.42,
    figsize=None,
    margins=(0.70, 0.25, 0.62, 0.65),
    xticks=None,
    yticks=None,
    tick_format="%.1f",
    ytick_rotation=90,
    title_fontsize=14,
    title_pad=23,
    xtick_fontsize=10,
    ytick_fontsize=10,
    xlabel_fontsize=14,
    ylabel_fontsize=14,
    xtick_pad=5,
    ytick_pad=7,
    xlabel_pad=6,
    ylabel_pad=7,
    font_family="DejaVu Sans",
    zline=True,
    distribution=True,
):
    """Assemble cached screenshots; all text, axes and guides remain vector.

    panel_height, panel_gap, figsize and margins=(left,right,bottom,top) are
    inches. Fonts and *_pad are points. figsize=None derives the canvas size;
    specifying figsize fits all three images while preserving physical aspect.
    Y ticks/spines are visible on all panels, with numeric labels only at left.

    zline/distribution: True uses reference-inspired default positions, False
    or None hides the annotation, and a dict overrides the corresponding
    add_*_guide keyword arguments. Guides do not identify features automatically.
    """
    ordered = [views[key] for key in ("desmin", "actin", "overlay")]
    extent = np.asarray(ordered[0].extent, dtype=float)
    if not all(np.allclose(view.extent, extent) for view in ordered):
        raise ValueError("All views must have identical calibrated bounds.")
    x0, x1, y0, y1 = extent
    aspect = (x1 - x0) / (y1 - y0)
    left, right, bottom, top = map(float, margins)
    if min(left, right, bottom, top, panel_gap) < 0 or panel_height <= 0:
        raise ValueError(
            "Margins/gap must be nonnegative; panel_height must be positive."
        )
    if figsize is None:
        fw = left + 3 * panel_height * aspect + 2 * panel_gap + right
        fh = bottom + panel_height + top
        start_x, start_y = left, bottom
    else:
        fw, fh = map(float, figsize)
        available_w, available_h = fw - left - right - 2 * panel_gap, fh - bottom - top
        if min(available_w, available_h) <= 0:
            raise ValueError("figsize is too small for the margins and gaps.")
        panel_height = min(available_h, available_w / (3 * aspect))
        start_x = left + (available_w - 3 * panel_height * aspect) / 2
        start_y = bottom + (available_h - panel_height) / 2
    width = panel_height * aspect
    colors = ordered[0].colors
    axes = []
    with (
        plt.style.context("default"),
        mpl.rc_context(
            {
                "font.family": font_family,
                "figure.autolayout": False,
                "figure.constrained_layout.use": False,
                "text.usetex": False,
            }
        ),
    ):
        fig = plt.figure(figsize=(fw, fh), facecolor="black")
        for i, (view, title) in enumerate(zip(ordered, ("Desmin", "Actin", "Overlay"))):
            ax = fig.add_axes(
                [
                    (start_x + i * (width + panel_gap)) / fw,
                    start_y / fh,
                    width / fw,
                    panel_height / fh,
                ],
                facecolor="black",
                sharex=axes[0] if axes else None,
                sharey=axes[0] if axes else None,
            )
            axes.append(ax)
            ax.imshow(
                view.image,
                extent=view.image_extent,
                origin="upper",
                interpolation="none",
                aspect="equal",
            )
            ax.set_xticks(np.linspace(x0, x1, 5) if xticks is None else xticks)
            ax.set_yticks(np.linspace(y0, y1, 5) if yticks is None else yticks)
            ax.set_xlim(x0, x1)
            ax.set_ylim(y0, y1)
            ax.xaxis.set_major_formatter(FormatStrFormatter(tick_format))
            ax.yaxis.set_major_formatter(FormatStrFormatter(tick_format))
            ax.tick_params(colors="white", direction="in", length=3, width=0.7)
            ax.tick_params(axis="x", labelsize=xtick_fontsize, pad=xtick_pad)
            ax.tick_params(
                axis="y",
                left=True,
                labelleft=(i == 0),
                labelsize=ytick_fontsize,
                pad=ytick_pad,
                labelrotation=ytick_rotation,
            )
            for side, spine in ax.spines.items():
                spine.set_visible(side in ("left", "bottom"))
                spine.set_color("#AAAAAA")
                spine.set_linewidth(0.7)
            for label in ax.get_xticklabels():
                label.set_ha("center")
            for value, label in zip(ax.get_yticks(), ax.get_yticklabels()):
                label.set_va("bottom" if np.isclose(value, y0) else "center")
            ax.set_title(
                title,
                color=colors.get(title.lower(), "white"),
                fontsize=title_fontsize,
                pad=title_pad,
            )
        if zline is not False and zline is not None:
            add_zline_guide(axes[0], **(zline if isinstance(zline, dict) else {}))
        if distribution is not False and distribution is not None:
            kw = {
                "color": colors["desmin"],
                **(distribution if isinstance(distribution, dict) else {}),
            }
            add_distribution_guide(axes[2], **kw)
        # Position shared labels using the real tick-label bounds.
        fig.canvas.draw()
        renderer = fig.canvas.get_renderer()
        xboxes = [
            t.get_window_extent(renderer) for ax in axes for t in ax.get_xticklabels()
        ]
        yboxes = [t.get_window_extent(renderer) for t in axes[0].get_yticklabels()]
        label_y = min(b.y0 for b in xboxes) / fig.bbox.height - xlabel_pad / (72 * fh)
        label_x = min(b.x0 for b in yboxes) / fig.bbox.width - ylabel_pad / (72 * fw)
        fig.supxlabel(
            "x (μm)",
            color="white",
            fontsize=xlabel_fontsize,
            va="top",
            x=(start_x + (3 * width + 2 * panel_gap) / 2) / fw,
            y=label_y,
        )
        fig.supylabel(
            "y (μm)",
            color="white",
            fontsize=ylabel_fontsize,
            ha="right",
            x=label_x,
            y=(start_y + panel_height / 2) / fh,
        )
    return fig, tuple(axes)

In [4]:
bbox = BoundingSquare(30, 15, 35)

img = AICSImage(datadir / "Image 4A- FED LatticeSIM.czi")

img_data = img.get_image_data("CXYZ", T=0)
pixel_sizes = (
    img.physical_pixel_sizes.X,
    img.physical_pixel_sizes.Y,
    img.physical_pixel_sizes.Z,
)

prepared = prepare_xy_pair(
    img_data,
    pixel_sizes,
    bbox=bbox,
    desmin_channel=0,
    actin_channel=1,
)

In [5]:
views = render_sim_triptych(
    prepared,
    height_px=1600,
    colors={"desmin": "#CC3311", "actin": "#0077BB"},
    color_modes={"desmin": "constant", "actin": "constant"},
    clim=(0, 0.2),
    opacity=(0, 0.1, 0.8),
)

2026-09-28 14:32:21.108 (  54.855s) [    14DE351B0740]vtkXOpenGLRenderWindow.:1416  WARN| bad X server connection. DISPLAY=


In [ ]:
fig, axes = compose_sim_triptych(
    views,
    panel_height=2.6,  # Inches
    panel_gap=0.42,  # Inches
    figsize=None,  # Or specify (width, height)
    margins=(0.70, 0.25, 0.62, 0.65),  # Left, right, bottom, top
    title_fontsize=14,
    xtick_fontsize=10,
    ytick_fontsize=10,
    xlabel_fontsize=14,
    ylabel_fontsize=14,
    zline={
        "coords": "data",  # Local μm coordinates
        "start": (26, 1),
        "end": (34.5, 12),
        "n_ticks": 8,
        "tick_length": 0.8,
        "text_position": (37, 1.7),
        "fontsize": 11,
    },
    distribution={
        "text_position": (0.62, 1.01),
        "arrow_tips": (
            (0.66, 1.015),
            (0.72, 1.015),
            (0.985, 1.015),
        ),  # Axes fractions by default
        "tail_offset": (5, 7),  # Points; controls tail length
        "mutation_scale": 10,  # Arrowhead size
        "fontsize": 11,
    },
)

In [52]:
savepath = "../../figures/4a"
save_panel(fig, savepath, formats=("svg",))
display(SVG(filename=str(savepath + ".svg")))
plt.close(fig)